
 ## Transform Constructors Data

 1. Read bronze `constructors` table
 1. Keep only the columns required for analytics (Drop `url` column)
 1. Standardise column names using snake_case (`constructorId` → `constructor_id`)
 1. Rename columns to make them more meaningful (`name` → `constructor_name`)
 1. Remove duplicate records
 1. Transform values of column `nationality` to Title Case
 1. Write the transformed data to silver `constructors` table


##Entity Relationship Diagram- Formula1 Bronze Schema
![formula1-silver-data-erd.png](./formula1-silver-data-erd.png "formula1-silver-data-erd.png")

In [0]:
%run ../00-common/1.Environment-Config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"


## Step 1 Read bronze constructors table

In [0]:
constructors_df = spark.read.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
constructors_dropped_df = constructors_df.drop("url")

In [0]:
display(constructors_dropped_df)

 #### Step 3 & 4 - Standardise Column Names
 - Standardise column names using snake_case (`constructorId` → `constructor_id`)
 - Rename columns to make them more meaningful (`name` → `constructor_name`)

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_renamed_df= (
    constructors_dropped_df
            .withColumnsRenamed({
                "constructorId": "constructor_id",
                "name": "constructor_name"
            })
)

In [0]:
display(constructors_renamed_df)

#### Step 5 - Remove duplicate records

In [0]:
#display duplicates
from pyspark.sql import functions as F

# Group by all columns (or pass a list of specific columns)
duplicate_counts = constructors_renamed_df.groupBy(constructors_renamed_df.columns) \
    .agg(F.count("*").alias("occurrence_count")) \
    .filter(F.col("occurrence_count") > 1)

duplicate_counts.show()



In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_distinct_df)

#### Step 6 - Transform values of column `nationality` to Title Case

In [0]:
constructors_finals_df = (
    constructors_distinct_df
    .withColumn('nationality',F.initcap(F.col("nationality")))
)

In [0]:
display(constructors_finals_df)

## Step 8 - Write the transformed data to silver `constructors` table

In [0]:
(
    constructors_finals_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))